In [1]:
%pip install highspy highspy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Импорты
from pathlib import Path
import numpy as np
import highspy
from time import perf_counter
import itertools

##### Идея решения:

1. Находим множества, в которых вершины не соеденены между собой
2. Составляем задачу линейного программирования и решаем методом ветвей и границ
3. Вызможно 4 исхода:
   1. Решение пустое — выходим из узла
   2. Решение меньше текущего оптимального — выходим из узла
   3. Решение целочисленное — проверяем, является ли оно кликой и либо обновляем оптимум, либо добавляем доп. ограничение
   4. Решение дробное — выбираем дробную переменную ближе к 0.5 (как наиболее неопределённое) и ветвимся по нему

In [ ]:
EPS = 1e-9


def read_graph(filepath):
    filepath = Path(filepath)
    n = None
    edges = []

    with filepath.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("c"):
                continue

            parts = line.split()
            if parts[0] == "p":
                n = int(parts[2])
            elif parts[0] == "e":
                u = int(parts[1]) - 1
                v = int(parts[2]) - 1
                if u != v:
                    edges.append((u, v))

    graph = [set() for _ in range(n)]
    for u, v in edges:
        graph[u].add(v)
        graph[v].add(u)

    return graph


def get_nonneighbors(graph):
    n = len(graph)
    all_vertices = set(range(n))
    nonneighbors = []

    for v in range(n):
        nn = all_vertices - graph[v] - {v}
        nonneighbors.append(nn)

    return nonneighbors


def is_clique(vertices, graph):
    verts = list(vertices)
    for i in range(len(verts)):
        u = verts[i]
        for j in range(i + 1, len(verts)):
            v = verts[j]
            if v not in graph[u]:
                return False
    return True


def check_solution(clique, graph):
    n = len(graph)

    if any(v < 0 or v >= n for v in clique):
        return False
    if len(set(clique)) != len(clique):
        return False

    return is_clique(clique, graph)


def get_independent_set_from_pair(u, v, graph, vert_order):
    indep = [u, v]
    forbidden = {u, v} | graph[u] | graph[v]

    for w in vert_order:
        if w not in forbidden:
            indep.append(w)
            forbidden.add(w)
            forbidden.update(graph[w])

    return tuple(sorted(indep))


def get_initial_independent_sets(graph):
    n = len(graph)
    nonneighbors = get_nonneighbors(graph)
    # Жадная эвристика
    vert_order = sorted(range(n), key=lambda x: len(graph[x]))

    covered_nonedges = set()
    indep_sets = []

    for u in range(n):
        for v in sorted(nonneighbors[u]):
            if u >= v:
                continue

            pair = (u, v)
            if pair in covered_nonedges:
                continue

            s = get_independent_set_from_pair(u, v, graph, vert_order)
            indep_sets.append(s)

            # Нет смысла перебирать все подпары независимого множества
            for a, b in itertools.combinations(s, 2):
                if b not in graph[a]:
                    covered_nonedges.add((min(a, b), max(a, b)))

    return indep_sets


def get_base_model(n: int, indep_sets):
    highs = highspy.Highs()
    highs.setMaximize()
    inf = highspy.kHighsInf

    # x_i in [0, 1]
    for _ in range(n):
        highs.addVar(0.0, 1.0)

    # max sum x_i
    for i in range(n):
        highs.changeColCost(i, 1.0)

    # sum_{i in S} x_i <= 1 for each independent set S
    for s in indep_sets:
        indices = np.array(s, dtype=np.int32)
        values = np.ones(len(s), dtype=np.double)
        highs.addRow(-inf, 1.0, len(s), indices, values)

    return highs


def solve_lp_on_node(
    base_model: highspy.Highs,
    fixed_zero: set,
    fixed_one: set,
):
    highs = highspy.Highs()
    highs.passModel(base_model.getLp())

    for i in fixed_zero:
        highs.changeColBounds(i, 0.0, 0.0)
    for i in fixed_one:
        highs.changeColBounds(i, 1.0, 1.0)

    highs.run()
    status = highs.getModelStatus()

    if status != highspy.HighsModelStatus.kOptimal:
        return None, None, status

    sol = highs.getSolution()
    info = highs.getInfo()

    x = np.array(sol.col_value, dtype=float)
    ub = float(info.objective_function_value)
    return x, ub, status

def is_integer_solution(x, eps = 1e-7):
    return np.all((x <= eps) | (x >= 1.0 - eps))


def get_active_vertices(x, eps = 1e-7):
    return [i for i, val in enumerate(x) if val >= 1.0 - eps]


def choose_branch_var(x, fixed_zero, fixed_one, graph):
    free = set(range(len(graph))) - fixed_zero - fixed_one
    best_i = None
    best_key = None

    for i in free:
        val = x[i]
        if EPS < val < 1.0 - EPS:
            nonneighbors_in_free = len(free - graph[i] - {i})
            key = (abs(val - 0.5), -nonneighbors_in_free, -len(graph[i]))
            if best_key is None or key < best_key:
                best_key = key
                best_i = i
    return best_i

def branch_and_bound(
    graph: dict,
    base_model: highspy.Highs,
    state: dict,
    fixed_zero: set,
    fixed_one: set,
    time_limit_sec: float,
):
    # Таймаут
    if (
        time_limit_sec is not None
        and perf_counter() - state["start_time"] > time_limit_sec
    ):
        state["timed_out"] = True
        return

    state["nodes_visited"] += 1
    n = len(graph)

    # Ранняя проверка совместимости уже выбранных вершин
    if not is_clique(fixed_one, graph):
        return

    # Кандидаты: вершины, которые еще не зафиксированы и смежны всем вершинам из fixed_one
    candidates = set(range(n)) - fixed_zero - fixed_one
    for u in fixed_one:
        candidates &= graph[u]

    # Дешевая граница: даже если взять всех кандидатов, текущий best не побить
    if len(fixed_one) + len(candidates) <= len(state["best_clique"]):
        return

    # Все некандидаты уже точно не могут попасть в клику -> эффективно фиксируем их в 0
    fixed_zero_eff = fixed_zero | ((set(range(n)) - fixed_one) - candidates)

    x, ub, status = solve_lp_on_node(base_model, fixed_zero_eff, fixed_one)
    if status != highspy.HighsModelStatus.kOptimal:
        return

    # Отсечение по верхней границе
    if ub <= len(state["best_clique"]) + 1e-9:
        return

    order = sorted(range(n), key=lambda i: x[i], reverse=True)
    clique = []
    for v in order:
        if v in fixed_zero_eff:
            continue
        if all(v in graph[u] for u in clique):
            clique.append(v)

    if len(clique) > len(state["best_clique"]):
        state["best_clique"] = clique

    if is_integer_solution(x):
        chosen = get_active_vertices(x)
        if is_clique(chosen, graph) and len(chosen) > len(state["best_clique"]):
            state["best_clique"] = chosen[:]
        return

    # Ветвление по дробной переменной
    k = choose_branch_var(x, fixed_zero, fixed_one, graph)
    if k is None:
        return

    # Пробуем x_k = 1
    fixed_one_2 = fixed_one.copy()
    fixed_one_2.add(k)
    branch_and_bound(graph, base_model, state, fixed_zero, fixed_one_2, time_limit_sec)

    # Пробуем x_k = 0
    fixed_zero_2 = fixed_zero.copy()
    fixed_zero_2.add(k)
    branch_and_bound(graph, base_model, state, fixed_zero_2, fixed_one, time_limit_sec)

def get_start_clique(graph):
    n = len(graph)
    order = sorted(range(n), key=lambda v: len(graph[v]), reverse=True)

    best = []
    for start in order:
        clique = [start]
        candidates = set(graph[start])

        while candidates:
            v = max(candidates, key=lambda x: len(graph[x] & candidates))
            clique.append(v)
            candidates &= graph[v]

        if len(clique) > len(best):
            best = clique

    return best

def run(filepath, time_limit_sec):
    graph = read_graph(filepath)

    indep_sets = get_initial_independent_sets(graph)
    base_model = get_base_model(len(graph), indep_sets)

    initial_clique = get_start_clique(graph)
    state = {
        "best_clique": initial_clique,
        "initial_clique_size": len(initial_clique),
        "nodes_visited": 0,
        "timed_out": False,
        "start_time": perf_counter(),
    }

    branch_and_bound(
        graph=graph,
        base_model=base_model,
        state=state,
        fixed_zero=set(),
        fixed_one=set(),
        time_limit_sec=time_limit_sec,
    )

    ok = check_solution(state["best_clique"], graph)

    print(f"File: {Path(filepath).name}")
    print("Best clique size:", len(state["best_clique"]))
    print("Best clique:", [v + 1 for v in state["best_clique"]])
    print("BnB helped:", len(state["best_clique"]) > state["initial_clique_size"])
    print("Nodes visited:", state["nodes_visited"])
    print("Timed out:", state["timed_out"])
    print("Checker:", "OK" if ok else "FAILED")
    print("-" * 40)
    return len(state["best_clique"])

In [ ]:
CHECK_OPT = {
    # Easy
    "C125.9.clq": 34,
    "johnson8-2-4.clq": 4,
    "johnson16-2-4.clq": 8,
    "MANN_a9.clq": 16,
    "keller4.clq": 11,
    "hamming8-4.clq": 16,
    # Moderate
    "brock200_1.clq": 21,
    "brock200_2.clq": 12,
    "brock200_3.clq": 15,
    "brock200_4.clq": 17,
    "gen200_p0.9_44.clq": 44,
    "gen200_p0.9_55.clq": 55,
    "MANN_a27.clq": 126,
    "p_hat1000-1.clq": 10,
    "p_hat1000-2.clq": 46,
    "p_hat300-3.clq": 36,
    "p_hat500-3.clq": 50,
    "sanr200_0.9.clq": 42,
    "sanr400_0.7.clq": 21,
}

np.set_printoptions(precision=4, suppress=True)

for filename, expected_opt in CHECK_OPT.items():
    found = run(Path(f"inputs/lab02/{filename}"), time_limit_sec=60)
    print(found == expected_opt,
        f"{Path(filename).name}: найден размер клики {found}, но ожидается {expected_opt}"
    )

File: C125.9.clq
Best clique size: 34
Best clique: [5, 54, 7, 45, 70, 114, 104, 125, 11, 98, 19, 80, 29, 31, 96, 49, 67, 9, 40, 44, 66, 110, 99, 121, 25, 34, 117, 77, 68, 55, 79, 122, 103, 52]
BnB helped: False
Nodes visited: 13799
Timed out: False
Checker: OK
----------------------------------------
True C125.9.clq: найден размер клики 34, но ожидается 34
File: johnson8-2-4.clq
Best clique size: 4
Best clique: [1, 6, 15, 28]
BnB helped: False
Nodes visited: 37
Timed out: False
Checker: OK
----------------------------------------
True johnson8-2-4.clq: найден размер клики 4, но ожидается 4
File: johnson16-2-4.clq
Best clique size: 8
Best clique: [1, 6, 15, 28, 45, 66, 104, 119]
BnB helped: False
Nodes visited: 29057
Timed out: True
Checker: OK
----------------------------------------
True johnson16-2-4.clq: найден размер клики 8, но ожидается 8
File: MANN_a9.clq
Best clique size: 16
Best clique: [10, 19, 22, 25, 2, 29, 9, 13, 5, 33, 16, 36, 6, 37, 40, 43]
BnB helped: False
Nodes visite